# Module: Fetching Census Data
## US Census Bureau API - Tehama County, CA
**Data:** API pulls real Demographic and Sociaeconomic Data for Tehama County (FIPS:06103)
**Data Source:** ACS 5-Year Estimates
**Comparision:** Tehama Vs CA State Average
**Author:** Abishek Heyer
**Date:** April 2026

In [1]:
#Libraries
import requests
import pandas as pd
import os
from dotenv import load_dotenv

In [2]:
load_dotenv()
CENSUS_API_KEY = os.getenv("CENSUS_API_KEY")


In [3]:
# Key variables
TEHAMA_FIPS ="06103" #Tehama County
STATE_FIPS= "06"     #CA
COUNTY_FIPS= "103"   #Tehama within CA
ACS_YEAR = 2024      #Year 

if CENSUS_API_KEY:
    print("Key loaded Successfully")
else:
    prnt("Key not found")

Key loaded Successfully


In [4]:
# Variables & Constants 

VARIABLES = {

    # Age & Sex
    "B01003_001E": "total_population",
    "B01001_002E": "male_population",
    "B01001_026E": "female_population",
    "B01002_001E": "median_age",
    "B01001_003E": "male_under_5",
    "B01001_027E": "female_under_5",
    "B01001_004E": "male_5_to_9",
    "B01001_005E": "male_10_to_14",
    "B01001_006E": "male_15_to_17",
    "B01001_028E": "female_5_to_9",
    "B01001_029E": "female_10_to_14",
    "B01001_030E": "female_15_to_17",
    "B01001_020E": "male_65_66",
    "B01001_021E": "male_67_69",
    "B01001_022E": "male_70_74",
    "B01001_023E": "male_75_79",
    "B01001_024E": "male_80_84",
    "B01001_025E": "male_85_plus",
    "B01001_044E": "female_65_66",
    "B01001_045E": "female_67_69",
    "B01001_046E": "female_70_74",
    "B01001_047E": "female_75_79",
    "B01001_048E": "female_80_84",
    "B01001_049E": "female_85_plus",

    # Race & Ethnicity
    "B03003_003E": "hispanic_latino",
    "B02001_002E": "white_alone",
    "B02001_003E": "black",
    "B02001_004E": "american indian_alone",
    "B02001_005E": "asian_alone",
    "B02001_006E": "nhpi_alone",
    "B02001_008E": "two_or_more_races",

    # Income & Poverty
    "B19013_001E": "median_household_income",
    "B19301_001E": "per_capita_income",
    "B17001_002E": "population_below_poverty",
    "B17001_001E": "poverty_universe",

    # Housing
    "B25001_001E": "housing_units",
    "B25003_002E": "owner_occupied_units",
    "B25003_001E": "occupied_housing_units",
    "B25077_001E": "median_home_value",
    "B25088_002E": "median_costs_with_mortgage",
    "B25088_003E": "median_costs_without_mortgage",
    "B25064_001E": "median_gross_rent",

    # Subject Tables (S prefix) 
    "S1101_C01_001E": "total_households",
    "S1101_C01_002E": "avg_household_size",     
    "S1501_C02_014E": "pct_hs_diploma_25plus",
    "S1501_C02_015E": "pct_bachelors_25plus",
    "S1810_C03_001E": "pct_disability",      
    
    # Data Profile Tables (DP prefix)
    "DP02_0114PE":    "pct_language_non_english",
    "DP03_0003E":     "labor_force_16plus",
    "DP03_0003PE":    "pct_labor_force_16plus",
    "DP03_0011E":     "female_labor_force_16plus",
    "DP03_0011PE":    "pct_female_labor_force_16plus",
    "DP03_0025E":     "mean_travel_time_to_work",
}

In [5]:
# Fetch Function (handles B, S, and DP tables)

def fetch_acs(year, geography):
    """
    Fetch ACS data from B, S, and DP table endpoints.
    
    B tables: available 2010-2024
    S tables: available 2015-2024
    DP tables: available 2015-2024
    
    For years before 2015, S and DP variables return NaN.
    """

    # Separate variables by table type
    b_vars  = {k: v for k, v in VARIABLES.items()
               if not k.startswith(("S", "DP"))}
    s_vars  = {k: v for k, v in VARIABLES.items()
               if k.startswith("S")}
    dp_vars = {k: v for k, v in VARIABLES.items()
               if k.startswith("DP")}

    #  Build geography parameters 
    if geography == "county":
        geo_param = (f"for=county:{COUNTY_FIPS}"
                     f"&in=state:{STATE_FIPS}")
        label = "Tehama County"
        fips  = TEHAMA_FIPS
    elif geography == "state":
        geo_param = f"for=state:{STATE_FIPS}"
        label = "California"
        fips  = STATE_FIPS
    elif geography == "nation":
        geo_param = "for=us:1"
        label = "United States"
        fips  = "US"

    result = {}

    #  Fetch B tables (available all years) 
    if b_vars:
        var_string = ",".join(["NAME"] + list(b_vars.keys()))
        url  = (f"https://api.census.gov/data/{year}/acs/acs5"
                f"?get={var_string}&{geo_param}"
                f"&key={CENSUS_API_KEY}")
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        data = resp.json()
        row  = dict(zip(data[0], data[1]))
        for k, v in b_vars.items():
            result[v] = row.get(k)

    # Fetch S tables (available 2015+)
    if s_vars:
        if year >= 2015:
            var_string = ",".join(["NAME"] + list(s_vars.keys()))
            url  = (f"https://api.census.gov/data/{year}/acs/acs5/subject"
                    f"?get={var_string}&{geo_param}"
                    f"&key={CENSUS_API_KEY}")
            try:
                resp = requests.get(url, timeout=30)
                resp.raise_for_status()
                data = resp.json()
                row  = dict(zip(data[0], data[1]))
                for k, v in s_vars.items():
                    result[v] = row.get(k)
            except Exception as e:
                print(f"S tables {year}: {e}")
                for v in s_vars.values():
                    result[v] = pd.NA
        else:
            # Before 2015 — set to NaN
            for v in s_vars.values():
                result[v] = pd.NA

    # ── Fetch DP tables (available 2015+) ─────────────────────
    if dp_vars:
        if year >= 2015:
            var_string = ",".join(["NAME"] + list(dp_vars.keys()))
            url  = (f"https://api.census.gov/data/{year}/acs/acs5/profile"
                    f"?get={var_string}&{geo_param}"
                    f"&key={CENSUS_API_KEY}")
            try:
                resp = requests.get(url, timeout=30)
                resp.raise_for_status()
                data = resp.json()
                row  = dict(zip(data[0], data[1]))
                for k, v in dp_vars.items():
                    result[v] = row.get(k)
            except Exception as e:
                print(f" DP tables {year}: {e}")
                for v in dp_vars.values():
                    result[v] = pd.NA
        else:
            # Before 2015 — set to NaN
            for v in dp_vars.values():
                result[v] = pd.NA


    df = pd.DataFrame([result])

    # Convert to numbers
    for col in VARIABLES.values():
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            df[col] = df[col].where(
                df[col] > -600000000, other=pd.NA)

    # Add metadata 
    df["geography"] = label
    df["geo_type"]  = geography
    df["acs_year"]  = year
    df["fips"]      = fips
    df["fetched_at"]= pd.Timestamp.now('UTC').isoformat()

    return df


print("fetch_acs() ready")
print("\nTable availability:")
print("  B tables  : 2010 → 2024 (all years)")
print("  S tables  : 2015 → 2024 (NaN for 2010-2014)")
print("  DP tables : 2015 → 2024 (NaN for 2010-2014)")
print(f"\nVariables loaded:")
print(f"  B : {len({k: v for k, v in VARIABLES.items() if not k.startswith(('S','DP'))})}")
print(f"  S : {len({k: v for k, v in VARIABLES.items() if k.startswith('S')})}")
print(f"  DP: {len({k: v for k, v in VARIABLES.items() if k.startswith('DP')})}")

fetch_acs() ready

Table availability:
  B tables  : 2010 → 2024 (all years)
  S tables  : 2015 → 2024 (NaN for 2010-2014)
  DP tables : 2015 → 2024 (NaN for 2010-2014)

Variables loaded:
  B : 42
  S : 5
  DP: 6


In [6]:
# Fetch Raw Data from 2010 to 2024 5 Year ACS
# Each call fetches all 40 variables at once

import time

YEARS = list(range(2010, 2025))
GEOGRAPHIES = ["county", "state", "nation"]
frames       = []
errors       = []

print("FETCHING ALL YEARS")
print("=" * 50)
print(f"Years      : {YEARS[0]} → {YEARS[-1]}")
print(f"Geographies: Tehama County, California, US")
print(f"Total calls: {len(YEARS) * len(GEOGRAPHIES)}")
print("=" * 50)

for geo in GEOGRAPHIES:
    print(f"\nFetching {geo}...")
    for year in YEARS:
        try:
            df = fetch_acs(year, geo)
            frames.append(df)
            print(f"{year}")
            time.sleep(0.5)
        except Exception as e:
            print(f"{year} failed: {e}")
            errors.append(f"{geo} {year}: {e}")

# ── Combine 
raw_df = pd.concat(frames, ignore_index=True)

print("\n" + "=" * 50)
print(f"   Fetch complete")
print(f"   Total rows    : {len(raw_df)}")
print(f"   Tehama rows   : {len(raw_df[raw_df['geography']=='Tehama County'])}")
print(f"   CA rows       : {len(raw_df[raw_df['geography']=='California'])}")
print(f"   US rows       : {len(raw_df[raw_df['geography']=='United States'])}")

if errors:
    print(f"\n{len(errors)} errors:")
    for e in errors:
        print(f"   {e}")

FETCHING ALL YEARS
Years      : 2010 → 2024
Geographies: Tehama County, California, US
Total calls: 45

Fetching county...
2010
2011
2012
2013
2014
2015
2016
2017
2018
2019
2020
2021
2022
2023
2024

Fetching state...
2010
2011
2012
2013
2014
2015
2016
2017
2018
2019
2020
2021
2022
2023
2024

Fetching nation...
2010
2011
2012
2013
2014
2015
2016
2017
2018
2019
2020
2021
2022
2023
2024

   Fetch complete
   Total rows    : 45
   Tehama rows   : 15
   CA rows       : 15
   US rows       : 15


In [7]:
# Save Raw Data 
# Save exactly what came from the API — no modification
from pathlib import Path

# Define save path 
RAW_DIR = Path("Data/raw/Census")
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Save the complete raw file as Paraquet and CSV
raw_path = RAW_DIR / "census_raw_all_geographies.parquet"
raw_df.to_parquet(raw_path, index=False)
csv_path = RAW_DIR / "census_raw_expanded.csv"
raw_df.to_csv(csv_path, index=False)

# Verify the file was actually created 
file_size = raw_path.stat().st_size

print("RAW DATA SAVED")
print("=" * 50)
print(f"  File    : {raw_path}")
print(f"  Size    : {file_size:,} bytes ({file_size/1024:.1f} KB)")
print(f"  Rows    : {len(raw_df)}")
print(f"  Columns : {len(raw_df.columns)}")

print(f"\nContents:")
for year in sorted(raw_df["acs_year"].unique()):
    for geo in raw_df[raw_df["acs_year"]==year]["geography"].unique():
        subset = raw_df[
            (raw_df["acs_year"]  == year) & 
            (raw_df["geography"] == geo)
        ]
        print(f"{geo} ({year}) → {len(subset)} row")

print(f"\nColumns saved:")
for col in raw_df.columns:
    val = raw_df[raw_df["geography"]=="Tehama County"][col].iloc[0]
    print(f"  {col:<40} {str(val)[:20]}")

RAW DATA SAVED
  File    : Data\raw\Census\census_raw_all_geographies.parquet
  Size    : 50,207 bytes (49.0 KB)
  Rows    : 45
  Columns : 58

Contents:
Tehama County (2010) → 1 row
California (2010) → 1 row
United States (2010) → 1 row
Tehama County (2011) → 1 row
California (2011) → 1 row
United States (2011) → 1 row
Tehama County (2012) → 1 row
California (2012) → 1 row
United States (2012) → 1 row
Tehama County (2013) → 1 row
California (2013) → 1 row
United States (2013) → 1 row
Tehama County (2014) → 1 row
California (2014) → 1 row
United States (2014) → 1 row
Tehama County (2015) → 1 row
California (2015) → 1 row
United States (2015) → 1 row
Tehama County (2016) → 1 row
California (2016) → 1 row
United States (2016) → 1 row
Tehama County (2017) → 1 row
California (2017) → 1 row
United States (2017) → 1 row
Tehama County (2018) → 1 row
California (2018) → 1 row
United States (2018) → 1 row
Tehama County (2019) → 1 row
California (2019) → 1 row
United States (2019) → 1 row
Tehama

In [8]:
df = pd.read_parquet("Data/raw/Census/census_raw_all_geographies.parquet")
df

,total_population,male_population,female_population,median_age,male_under_5,female_under_5,male_5_to_9,male_10_to_14,male_15_to_17,female_5_to_9,...,labor_force_16plus,pct_labor_force_16plus,female_labor_force_16plus,pct_female_labor_force_16plus,mean_travel_time_to_work,geography,geo_type,acs_year,fips,fetched_at
0,62575,31198,31377,39.2,2216,2085,1993,2487,1646,2238,...,NaN,NaN,NaN,NaN,NaN,Tehama County,county,2010,06103,2026-04-22T17:52:27.951561+00:00
1,62985,31320,31665,39.4,2240,2095,1886,2564,1554,2195,...,NaN,NaN,NaN,NaN,NaN,Tehama County,county,2011,06103,2026-04-22T17:52:29.909961+00:00
2,63200,31428,31772,39.6,2199,2086,2045,2429,1467,2160,...,NaN,NaN,NaN,NaN,NaN,Tehama County,county,2012,06103,2026-04-22T17:52:31.930444+00:00
3,63241,31435,31806,39.9,2221,2048,2038,2335,1454,1974,...,NaN,NaN,NaN,NaN,NaN,Tehama County,county,2013,06103,2026-04-22T17:52:33.915514+00:00
4,63284,31494,31790,40.0,2232,1936,2266,2108,1379,2053,...,NaN,NaN,NaN,NaN,NaN,Tehama County,county,2014,06103,2026-04-22T17:52:35.920612+00:00
5,63152,31489,31663,40.5,2153,1865,2325,2060,1326,2026,...,26638.0,53.6,12337.0,48.9,24.6,Tehama County,county,2015,06103,2026-04-22T17:52:40.244512+00:00
6,63015,31379,31636,40.6,2027,1832,2527,1969,1262,1959,...,26434.0,53.3,12279.0,48.6,24.2,Tehama County,county,2016,06103,2026-04-22T17:52:44.498728+00:00
7,63247,31452,31795,41.1,2008,1868,2339,2166,1244,1895,...,26541.0,53.4,12249.0,48.2,23.5,Tehama County,county,2017,06103,2026-04-22T17:52:49.236155+00:00
8,63373,31504,31869,40.9,1980,1887,2293,2213,1229,1990,...,26749.0,53.5,12356.0,48.6,23.4,Tehama County,county,2018,06103,2026-04-22T17:52:53.487192+00:00
9,63912,31773,32139,41.0,1938,1901,2209,2351,1241,1780,...,26951.0,53.3,12589.0,49.0,23.4,Tehama County,county,2019,06103,2026-04-22T17:52:57.741639+00:00
